# Week 4 — Python to Bash Code Generator

Convert Python code to high-performance, correct Bash. Uses OpenAI, Anthropic, or Ollama (local).

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import gradio as gr

In [ ]:
MODEL_OPENAI = "gpt-4o-mini"
MODEL_ANTHROPIC = "claude-3-5-sonnet-20241022"
MODEL_OLLAMA = "llama3.2"

In [ ]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if openai_key:
    print(f"OpenAI key found (starts with {openai_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")
if anthropic_key:
    print(f"Anthropic key found (starts with {anthropic_key[:7]}...)")
else:
    print("ANTHROPIC_API_KEY not set")

openai_client = OpenAI()
anthropic_client = anthropic.Anthropic()
ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
print("Clients ready.")

In [ ]:
SYSTEM_PROMPT = """You convert Python code into high-performance Bash script.

Rules:
- Output only the Bash script. No explanations, no markdown code fences.
- The Bash must be correct, avoid errors, and run efficiently.
- Use standard POSIX/bash constructs; keep it simple and robust."""

In [ ]:
def generate_bash(python_code, model_choice):
    if not python_code or not python_code.strip():
        return "Paste Python code to convert."
    user_msg = f"Convert this Python code to Bash. Output only the Bash script.\n\n```python\n{python_code.strip()}\n```"
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]

    try:
        if model_choice == "OpenAI (gpt-4o-mini)":
            r = openai_client.chat.completions.create(model=MODEL_OPENAI, messages=messages, temperature=0.2)
            raw = (r.choices[0].message.content or "").strip()
        elif model_choice == "Anthropic (claude-3-5-sonnet)":
            msg = anthropic_client.messages.create(
                model=MODEL_ANTHROPIC,
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}],
            )
            raw = (msg.content[0].text if msg.content else "").strip()
        else:
            r = ollama_client.chat.completions.create(model=MODEL_OLLAMA, messages=messages, temperature=0.2)
            raw = (r.choices[0].message.content or "").strip()

        if raw.startswith("```"):
            raw = re.sub(r"^```\w*\n", "", raw)
            raw = re.sub(r"\n```\s*$", "", raw)
        return raw
    except Exception as e:
        return f"Error: {e}"

In [ ]:
with gr.Blocks(title="Python to Bash") as demo:
    gr.Markdown("## Python → Bash Code Generator")
    code_in = gr.Textbox(label="Python code", placeholder="Paste Python code here", lines=12)
    model_dropdown = gr.Dropdown(
        choices=["OpenAI (gpt-4o-mini)", "Anthropic (claude-3-5-sonnet)", "Ollama (llama3.2)"],
        value="OpenAI (gpt-4o-mini)",
        label="Model",
    )
    btn = gr.Button("Generate Bash", variant="primary")
    code_out = gr.Textbox(label="Generated Bash", lines=14)

    btn.click(fn=generate_bash, inputs=[code_in, model_dropdown], outputs=code_out)

demo.launch()